# TRAITEMENT DES DONNES FUSIONNEES DE LA PREMIERE FEUILLE EXCEL

**Objectif** : Traitement/apurement des variables démographiques
* Date de naissance 
* Sexe
* Situation Matrimoniale
* Nombre d'enfants

Nous aurons également pour mission de créer de nouvelles variables à partir des informations disponibles. Il s'agit du statut dans l'emploi (Fonctionnaire ou contractuelle), l'âge, l'age de première prise de service, l'ancienneté dans l'organisme, nombre d'année d'expérience.

Un contrôle sera effectué sur le rattachement administratif:

* Matricule
* Fonction
* Services
* Organismes
* Situation dans l'emploi
* Emplois
* Fonction
* Classes/Echelons
* Lieu d'affectation

## Chargement des packages necessaires

In [1]:
pip install fastparquet

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Paramètres
import io
import pandas as pd
# import boto3
import io
import unicodedata
import re
import numpy as np
import unicodedata
import unicodedata, re
import re

# FONCTIONS NECESSAIRES

### CREATION DE LA VARIABLE CATEGORIE 1 ET 2 

In [3]:
def build_categories(
    df: pd.DataFrame,
    grade_col="GRADE",
    grade2_col="GRADE 2",
    compute_categorie2=True,     # calcule CATEGORIE_2 depuis GRADE 2 si la colonne existe
    overwrite_categorie2=True    # réécrit CATEGORIE_2 même si déjà présente
):
    """
    - CATEGORIE_1 depuis GRADE (texte):
        'Non Fonctionnaire' si le texte contient 'non fonctionnaire' (insensible à la casse/espaces)
        sinon extrait la lettre A-D depuis 'Catégorie A7', 'Catégorie B', etc.
        NaN -> 'Non Fonctionnaire'
    - CATEGORIE_2 depuis GRADE 2 (lettres A-D):
        'A7' -> 'A', 'B' -> 'B', autres -> NaN -> 'Non Fonctionnaire'
    - Les 2 colonnes sont typées 'category' avec les mêmes modalités ordonnées.

    Retourne: df (copie) avec CATEGORIE_1 (et CATEGORIE_2 si demandé).
    """

    out = df.copy()
    cat_order = ["Non Fonctionnaire", "A", "B", "C", "D"]

    # ---------- helpers ----------
    def _cat_from_grade2_letter(val):
        """ Cette fonction transforme un GRADE 2 en catégorie """
        if pd.isna(val):
            return pd.NA  # si valeur manquante, on retourne pd.NA.
        s = str(val).strip().upper()  # on convertit en chaîne, on enlève les espaces et on met en majuscules
        if re.fullmatch(r"[ABCD]\d+", s):   # A7, B3, C1, D2...
            return s[0]
        if re.fullmatch(r"[ABCD]", s):      # A, B, C, D
            return s
        return pd.NA

    def _parse_cat_from_grade_text(val):
        if pd.isna(val):
            return pd.NA
        t = str(val)
        # ex.: "Non   fonctionnaire", "non-fonctionnaire"
        if re.search(r"\bnon\s*fonctionnaire\b", t, flags=re.I):
            return "Non Fonctionnaire"
        # ex.: "Catégorie A", "Categorie B7", "catégorie c titulaire"
        m = re.search(r"Cat[ée]gorie\s+([A-D])(?:\s*\d+)?", t, flags=re.I)
        if m:
            return m.group(1).upper()
        return pd.NA

    # ---------- CATEGORIE_1 depuis GRADE ----------
    if grade_col not in out.columns:
        raise KeyError(f"Colonne '{grade_col}' absente.")
    out["CATEGORIE_1"] = out[grade_col].apply(_parse_cat_from_grade_text)
    out["CATEGORIE_1"] = out["CATEGORIE_1"].fillna("Non Fonctionnaire") # Remplace les NaN par "Non Fonctionnaire"
    out["CATEGORIE_1"] = pd.Categorical(out["CATEGORIE_1"], categories=cat_order, ordered=True) #Transforme la colonne en type pd.Categorical avec l’ordre défini (cat_order)

    # ---------- (optionnel) CATEGORIE_2 depuis GRADE 2 ----------
    if compute_categorie2 and (grade2_col in out.columns):
        if overwrite_categorie2 or ("CATEGORIE_2" not in out.columns):
            cat2_raw = out[grade2_col].apply(_cat_from_grade2_letter)
            cat2 = cat2_raw.astype(object)
            cat2[pd.isna(cat2)] = "Non Fonctionnaire" # Remplace les NaN par "Non Fonctionnaire"
            out["CATEGORIE_2"] = pd.Categorical(cat2, categories=cat_order, ordered=True) # #Transforme la colonne en type pd.Categorical avec l’ordre défini (cat_order)

    return out 

### CREATION DE LA VARIABLE STATUT

In [4]:
def build_statut_from_cats(df: pd.DataFrame, 
                           emploi_col: str = "EMPLOI_NORM",
                           cat1_col: str | None = None,
                           cat1_candidates: tuple[str,str] = ("CATEGORIE_1", "CATEGORIE 1"),
                           cat2_col: str | None = None,
                           cat2_candidates: tuple[str,str] = ("CATEGORIE_2", "CATEGORIE 2"),
                           label_cas: str = "Cas particulier"):

    import unicodedata, re
    import numpy as np
    import pandas as pd

    out = df.copy()

    # --- Trouver les colonnes ---
    def _pick_col(cand_list):
        for c in cand_list:
            if c in out.columns:
                return c
        return None

    if cat1_col is None:
        cat1_col = _pick_col(cat1_candidates)
    if cat2_col is None:
        cat2_col = _pick_col(cat2_candidates)

    for col, name in zip([cat1_col, cat2_col, emploi_col], ["CATEGORIE_1","CATEGORIE_2","EMPLOI_NORM"]):
        if col is None or col not in out.columns:
            raise KeyError(f"Colonne '{name}' introuvable.")

    # --- EMPLOI renseigné ---
    emp_norm = out[emploi_col].fillna("").astype(str).str.strip()
    has_emploi = emp_norm.ne("")

    # --- Normalisation catégories ---
    def _norm_ascii_lower(x):
        if pd.isna(x):
            return ""
        s = str(x).strip()
        s = unicodedata.normalize("NFKD", s)
        s = s.encode("ascii", "ignore").decode("ascii")
        s = re.sub(r"\s+", " ", s).strip().lower()
        return s

    c1_norm = out[cat1_col].map(_norm_ascii_lower)
    c2_norm = out[cat2_col].map(_norm_ascii_lower)

    c1_is_nf = c1_norm.str.contains(r"\bnon\s*fonctionnaire\b", na=False)
    c2_is_nf = c2_norm.str.contains(r"\bnon\s*fonctionnaire\b", na=False)

    c1_is_abcd = c1_norm.str.fullmatch(r"[abcd]", na=False)
    c2_is_abcd = c2_norm.str.fullmatch(r"[abcd]", na=False)

    is_nf_any = c1_is_nf | c2_is_nf
    is_nf_all = c1_is_nf & c2_is_nf
    is_abcd_any = c1_is_abcd | c2_is_abcd
    is_abcd_all = c1_is_abcd & c2_is_abcd

    # --- STATUT ---
    statut = np.select(
        [(has_emploi & is_nf_any) | (~has_emploi & is_abcd_any),
         has_emploi & is_abcd_all,
         ~has_emploi & is_nf_all],
        [label_cas, "Fonctionnaire", "Non Fonctionnaire"],
        default="Non Fonctionnaire"
    ).astype(object)

    out["STATUT"] = pd.Categorical(
        statut,
        categories=["Non Fonctionnaire", "Fonctionnaire", label_cas],
        ordered=True
    )

    # --- Rapport automatique ---
    rep = {
        "repartition_statut": out["STATUT"].value_counts().reindex(
            ["Non Fonctionnaire","Fonctionnaire",label_cas], fill_value=0
        )
    }

    return out, rep

### CREATION DE LA VARIABLE STATUT DEFINITIF 

Créer une variable Statut_def qui renseigne le statut définitif de l'individu sur une période donnée. En effet, un individu peut être fonctionnaire sur un organisme et ne pas contractuel sur un autre organisme. Le matricule permet d'identifier de manière unique un individu. On combinant le matricule et le code organisme on peut retrouver le statut de l'individu sur chaque organisme où il intervient. Pour une période de collecte donnée (mois et année), le statut définitif sera qu'il est fonctionnaire si sur au moins un organisme l'individu est fonctionnaire sinon non fonctionnaire. Attention, le contrôle se fait sur la période. Ici nous avons 72 périodes (janvier 2015 à décembre 2020).

In [5]:
def build_statut_def_periode(
    df: pd.DataFrame,
    statut_col: str = "STATUT",
    matricule_col: str = "MATRICULE",
    periode_col: str = "PERIODE",
    output_col: str = "Statut_def_mois",
    return_report: bool = True
):

    """
        Règle:
      - Pour chaque groupe (MATRICULE et PERIODE ), on définit le statut final:
          * Si au moins une ligne = 'Fonctionnaire' → Statut_def = 'Fonctionnaire'
          * Sinon si au moins une ligne = 'Cas particulier' → Statut_def = 'Cas particulier'
          * Sinon → 'Non Fonctionnaire'
          
          Colonnes attendues: statut_col, matricule_col, periode_col
    """
    out = df.copy()

    # Vérifications
    for col in [statut_col, matricule_col, periode_col]:
        if col not in out.columns:
            raise KeyError(f"Colonne '{col}' introuvable.")

    # Normalisation
    stat = out[statut_col].astype(str).str.strip().str.casefold()
    est_fonctionnaire = stat == "fonctionnaire"
    est_cas_particulier = stat == "cas particulier"

    # Agrégation par groupe. On regroupe par MATRICULE et PERIODE pour vérifier si au moins
    # une ligne est “Fonctionnaire” ou “Cas particulier”.
    key_cols = [matricule_col, periode_col]
    any_fonct = est_fonctionnaire.groupby([out[c] for c in key_cols]).transform("any")
    any_cas = est_cas_particulier.groupby([out[c] for c in key_cols]).transform("any")

    # Priorité pour le statut final
    out[output_col] = np.select(
        [any_fonct, any_cas],
        ["Fonctionnaire", "Cas particulier"],
        default="Non Fonctionnaire"
    )

    out[output_col] = pd.Categorical(
        out[output_col],
        categories=["Non Fonctionnaire", "Cas particulier", "Fonctionnaire"],
        ordered=True
    )

    if return_report:
        grp_summary = out.groupby(key_cols)[output_col].first()
        rep = {
            "Statut définifitif": out[output_col].value_counts()
                                   .reindex(["Non Fonctionnaire","Cas particulier","Fonctionnaire"], fill_value=0),
        }
        return out, rep

    return out

In [6]:
def build_statut_def_annee(df, 
                            statut_col="STATUT", 
                            matricule_col="MATRICULE",
                            date_collecte_col="DATE_COLLECTE",
                            output_col="Statut_def_annee",
                            return_report=True):
    """
    Détermine le statut définitif par MATRICULE et ANNEE (extraite de DATE_COLLECTE).
    
    Règle:
      - Si au moins une ligne = 'Fonctionnaire' → Statut_def = 'Fonctionnaire'
      - Sinon si au moins une ligne = 'Cas particulier' → Statut_def = 'Cas particulier'
      - Sinon → 'Non Fonctionnaire'
    
    Retour:
        - df mis à jour
        - report (distribution par Statut_def) si return_report=True
    """
    out = df.copy()

    # Vérifications colonnes
    for col in [statut_col, matricule_col, date_collecte_col]:
        if col not in out.columns:
            raise KeyError(f"Colonne '{col}' introuvable.")

    # Extraire l'année
    out[date_collecte_col] = pd.to_datetime(out[date_collecte_col], errors="coerce")
    out["ANNEE"] = out[date_collecte_col].dt.year

    # Normalisation du statut
    stat = out[statut_col].astype(str).str.strip().str.casefold()
    est_fonctionnaire = stat == "fonctionnaire"
    est_cas_particulier = stat == "cas particulier"

    # Regroupement par matricule et année
    key_cols = [matricule_col, "ANNEE"]
    any_fonct = est_fonctionnaire.groupby([out[c] for c in key_cols]).transform("any")
    any_cas = est_cas_particulier.groupby([out[c] for c in key_cols]).transform("any")

    # Détermination du statut définitif
    out[output_col] = np.select(
        [any_fonct, any_cas],
        ["Fonctionnaire", "Cas particulier"],
        default="Non Fonctionnaire"
    )

    out[output_col] = pd.Categorical(
        out[output_col],
        categories=["Non Fonctionnaire", "Cas particulier", "Fonctionnaire"],
        ordered=True
    )

    if return_report:
        rep = {
            "Statut_def_distribution": out[output_col].value_counts()
                                   .reindex(["Non Fonctionnaire","Cas particulier","Fonctionnaire"], fill_value=0)
        }
        return out, rep

    return out


### CREATION DE LA VARIABLE SEXE IMPUTE

In [19]:
def imputer_sexe(
    df: pd.DataFrame,
    sex_col="SEXE",               # Colonne brute contenant le sexe
    collect_col="DATE_COLLECTE",  # Colonne de date de collecte
    sex_valid_col="SEXE_IMPUTE",  # Colonne qui sera créée pour stocker les valeurs imputées
    debug=True                     # Affichage automatique par défaut
):
    """
    Impute les valeurs manquantes de sexe en utilisant le mode calculé
    au niveau ANNEE_COLLECTE × MOIS_COLLECTE.

    Paramètres :
        df : pd.DataFrame
            DataFrame à traiter
        sex_col : str
            Colonne brute du sexe
        collect_col : str
            Colonne de date de collecte
        sex_valid_col : str
            Nom de la colonne imputée à créer
        debug : bool
            Si True, affichage automatique des statistiques avant/après

    Retour :
        df : DataFrame avec la colonne imputée
        report : dictionnaire avec tableaux de statistiques
    """
    import pandas as pd
    import numpy as np

    # ---- Travail sur une copie pour ne pas modifier l'original ----
    df = df.copy()

    # ---- Conversion de la colonne date ----
    df[collect_col] = pd.to_datetime(df[collect_col], errors="coerce")

    # ---- Création des colonnes année et mois de collecte ----
    df["ANNEE_COLLECTE"] = df[collect_col].dt.year
    df["MOIS_COLLECTE"] = df[collect_col].dt.month

    # ---- Initialisation de la colonne imputée avec les valeurs existantes ----
    df[sex_valid_col] = df[sex_col]

    # ---- Calcul du mode du sexe par groupe (année × mois) ----
    mode_sexe_groupes = (
        df.dropna(subset=[sex_col])  # On ignore les NaN
          .groupby(["ANNEE_COLLECTE", "MOIS_COLLECTE"])[sex_col]
          .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
    )

    # ---- Imputation : remplacer NaN par le mode du groupe ----
    df[sex_valid_col] = df.apply(
        lambda row: mode_sexe_groupes.get(
            (row["ANNEE_COLLECTE"], row["MOIS_COLLECTE"]), row[sex_valid_col]
        ) if pd.isna(row[sex_valid_col]) else row[sex_valid_col],
        axis=1
    )

    # ---- Statistiques avant/après imputation ----
    tab_sexe_avant = df[sex_col].value_counts(dropna=False).sort_index()
    tab_sexe_apres = df[sex_valid_col].value_counts(dropna=False).sort_index()
    tab_sexe_apres_pct = df[sex_valid_col].value_counts(normalize=True, dropna=False).sort_index() * 100

    # ---- Tableau croisé avant/après ----
    crosstab_sexe = pd.crosstab(
        df[sex_col],
        df[sex_valid_col],
        dropna=False,
        margins=True,
        margins_name="Total"
    )

    # ---- Création du dictionnaire de report ----
    report = {
        "tab_sexe_avant": tab_sexe_avant,
        "tab_sexe_apres": tab_sexe_apres,
        "tab_sexe_apres_pct": tab_sexe_apres_pct,
        "crosstab_sexe": crosstab_sexe
    }

    # ---- Affichage automatique si debug=True ----
    if debug:
        print("=== Statistiques avant imputation ===")
        display(tab_sexe_avant.to_frame("Effectif"))
        print("\n=== Statistiques après imputation ===")
        display(tab_sexe_apres.to_frame("Effectif"))
        print("\n=== Pourcentages après imputation ===")
        display(tab_sexe_apres_pct.to_frame("Pourcentage (%)"))
        print("\n=== Tableau croisé avant/après ===")
        display(crosstab_sexe)

    return df, report

In [8]:
def verifier_variation_sexe(df, id_col="MATRICULE", sex_col="SEXE_IMPUTE", collect_col="DATE_COLLECTE"):
    """
    Vérifie si le sexe d'un individu varie dans le temps et met à jour SEXE_IMPUTE
    avec la valeur la plus récente selon DATE_COLLECTE.

    Retourne :
      - variation : DataFrame des individus pour lesquels le sexe n'est pas constant
      - details : DataFrame détaillé de ces individus
      - df : DataFrame mis à jour avec SEXE_IMPUTE actualisé
      - stats : dictionnaire avec tableaux avant/après
    """
    df = df.copy()
    
    # Conversion en datetime si nécessaire
    df[collect_col] = pd.to_datetime(df[collect_col], errors="coerce")
    
    # Tableau avant
    tab_avant = df[sex_col].value_counts(dropna=False).sort_index()
    
    # Nombre de valeurs distinctes de sexe par individu
    variation = df.groupby(id_col)[sex_col].nunique().reset_index()
    variation = variation.rename(columns={sex_col: "nb_valeurs_distinctes"})
    
    # Sélection des individus avec plus d'une valeur distincte
    variation = variation[variation["nb_valeurs_distinctes"] > 1]
    
    # Détails pour inspection
    details = df[df[id_col].isin(variation[id_col])].sort_values([id_col, collect_col])
    
    print(f"Nombre d'individus dont le sexe varie dans le temps : {variation.shape[0]}")

    # Mettre à jour SEXE_IMPUTE : prendre la valeur la plus récente
    latest_sex = df.sort_values([id_col, collect_col]).groupby(id_col, as_index=False).last()[[id_col, sex_col]]
    df = df.drop(columns=[sex_col]).merge(latest_sex, on=id_col, how="left")

    # Tableau après
    tab_apres = df[sex_col].value_counts(dropna=False).sort_index()
    tab_apres_pct = df[sex_col].value_counts(normalize=True, dropna=False).sort_index() * 100

    stats = {
        "tab_avant": tab_avant,
        "tab_apres": tab_apres,
        "tab_apres_pct": tab_apres_pct
    }

    return variation, details, df, stats


### CREATION DE LA VARIABLE AGE IMPUTE

In [9]:
def build_age_valide(
    df: pd.DataFrame,
    matricule_col="MATRICULE",
    birth_col="DATE NAISSANCE",
    collect_col="DATE_COLLECTE",
    sex_col="SEXE_IMPUTE",                  # ⚡️ ajouter le sexe
    age_min=18, age_max=70,
    do_impute_age=True
):
    """
    Construit les variables d'âge à partir des dates de naissance.
    Toujours : on prend la date la plus récente pour chaque matricule.

    Étapes :
      1) Nettoyage dates collecte et naissance
      2) Correction date naissance → la plus récente disponible par matricule
      3) Calcul AGE, AGE_VALIDE
      4) Option : imputation AGE_VALIDE manquant → AGE_IMPUTE
        (imputation par médiane par année, mois et sexe)
    """

    df = df.copy()

    # -------- Utilitaires --------
    def to_datetime_mixed(s):
        try:
            return pd.to_datetime(s, errors="coerce", format="mixed", dayfirst=True)
        except TypeError:
            return pd.to_datetime(s, errors="coerce", dayfirst=True, infer_datetime_format=True)

    def clean_birthdates(series):
        s = series.astype(str).str.strip().str.lower()
        time_only = s.str.fullmatch(r"\d{1,2}:\d{2}(:\d{2})?")
        zero_date = s.str.fullmatch(r"(0{1,2}[/\-]0{1,2}[/\-]0{2,4}|0000-00-00)")
        empties   = s.isin(["", "nan", "nat", "none", "nul", "null"])
        s = series.mask(time_only | zero_date | empties, np.nan)
        dt = to_datetime_mixed(s)

        # Numéros Excel
        serial = pd.to_numeric(series, errors="coerce")
        serial_mask = dt.isna() & serial.notna() & serial.between(1, 60000)
        if serial_mask.any():
            dt_serial = pd.to_datetime(serial[serial_mask], unit="D", origin="1899-12-30", errors="coerce")
            dt.loc[serial_mask] = dt_serial
        return dt

    # -------- 0) DATE_COLLECTE --------
    df[collect_col] = to_datetime_mixed(df[collect_col])
    if "ANNEE_COLLECTE" not in df.columns:
        df["ANNEE_COLLECTE"] = df[collect_col].dt.year
    if "MOIS_COLLECTE" not in df.columns:
        df["MOIS_COLLECTE"] = df[collect_col].dt.month

    # -------- 1) Nettoyage DATE NAISSANCE --------
    df["DATE_NAISSANCE_CLEAN"] = clean_birthdates(df[birth_col])
    df["ANNEE_NAISSANCE"] = df["DATE_NAISSANCE_CLEAN"].dt.year
    df["MOIS_NAISSANCE"]  = df["DATE_NAISSANCE_CLEAN"].dt.month
    df["JOUR_NAISSANCE"]  = df["DATE_NAISSANCE_CLEAN"].dt.day

    # -------- 2) DATE_NAISSANCE_CORRIGEE = date la + récente --------
    tmp = (
        df.dropna(subset=["DATE_NAISSANCE_CLEAN"])
          .sort_values([matricule_col, collect_col], ascending=[True, False])
          .drop_duplicates(subset=[matricule_col], keep="first")
          [[matricule_col, "DATE_NAISSANCE_CLEAN"]]
          .rename(columns={"DATE_NAISSANCE_CLEAN": "DATE_NAISSANCE_CORRIGEE"})
    )
    df = df.drop(columns=["DATE_NAISSANCE_CORRIGEE"], errors="ignore")
    df = df.merge(tmp, on=matricule_col, how="left", validate="many_to_one")
    df["DATE_NAISSANCE_CORRIGEE"] = pd.to_datetime(df["DATE_NAISSANCE_CORRIGEE"], errors="coerce")
    df["ANNEE_NAISSANCE_CORRIGEE"] = df["DATE_NAISSANCE_CORRIGEE"].dt.year
    df["MOIS_NAISSANCE_CORRIGEE"]  = df["DATE_NAISSANCE_CORRIGEE"].dt.month
    df["JOUR_NAISSANCE_CORRIGEE"]  = df["DATE_NAISSANCE_CORRIGEE"].dt.day

    # -------- 3) Calcul AGE --------
    birth = df["DATE_NAISSANCE_CORRIGEE"]
    ref   = df[collect_col]
    age = pd.Series(pd.NA, index=df.index, dtype="Float64")
    mask = birth.notna() & ref.notna()
    if mask.any():
        ydiff = ref[mask].dt.year - birth[mask].dt.year
        before_bday = (ref[mask].dt.month < birth[mask].dt.month) | (
            (ref[mask].dt.month == birth[mask].dt.month) & (ref[mask].dt.day < birth[mask].dt.day)
        )
        age.loc[mask] = (ydiff - before_bday.astype(int)).astype("Float64")
    df["AGE"] = age

    # -------- 4) AGE_VALIDE --------
    df["AGE_VALIDE"] = df["AGE"].where((df["AGE"].ge(age_min)) & (df["AGE"].le(age_max)))

    # -------- 5) AGE_IMPUTE --------
    if do_impute_age:
        # clés : année, mois et sexe
        group_keys = ["ANNEE_COLLECTE", "MOIS_COLLECTE"]
        if sex_col in df.columns:
            group_keys.append(sex_col)

        med = df.groupby(group_keys)["AGE_VALIDE"].transform("median")
        global_med = df["AGE_VALIDE"].median()
        df["AGE_IMPUTE"] = df["AGE_VALIDE"].where(df["AGE_VALIDE"].notna(), med.fillna(global_med)).round(0).astype("Int64")

    return df

### CREATION DE LA VARIABLE AGE DE PRISE DE SERVICE 


In [10]:
def build_age_service(
    df: pd.DataFrame,
    matricule_col="MATRICULE",
    start_col_raw="PRISE DE SERVICE",          # colonne brute (texte/mixtes)
    start_col_corrected="PRISE_DE_SERVICE_CORRIGEE",
    collect_col="DATE_COLLECTE",
    sex_col="SEXE_IMPUTE",                     # utiliser le sexe imputé
    year_collect_col="ANNEE_COLLECTE",
    month_collect_col="MOIS_COLLECTE",
    recompute_corrected=True,                  # recalcule PRISE_DE_SERVICE_CORRIGEE depuis collecte récente
    age_min=18,
    age_max=40,
    days_per_year=365,                         # fidèle à ton // 365 ; mettre 365.25 si besoin
    return_tables=True,
    sample_anomalies=10                        # nb de lignes à montrer pour l’aperçu anomalies
):
    """
    Étapes intégrées :
      1) Nettoie PRISE DE SERVICE (formats mixtes, placeholders, numéros Excel).
      2) Diagnostic : nb de dates uniques par MATRICULE, liste des matricules 'à problème'.
      3) Calcule PRISE_DE_SERVICE_CORRIGEE = valeur observée à la collecte la + récente par MATRICULE.
      4) Calcule AGE_AU_SERVICE, AGE_AU_SERVICE_VALIDE (bornage [age_min, age_max]).
      5) Impute AGE_AU_SERVICE_IMPUTE par médiane (ANNEE_COLLECTE × MOIS_COLLECTE × SEXE_IMPUTE si dispo).
      6) Retourne df enrichi + report (diagnostics et tableaux).
    """
    import pandas as pd
    import numpy as np

    df = df.copy()

    # ---------- utilitaires ----------
    def _to_datetime_mixed(s):
        try:
            return pd.to_datetime(s, errors="coerce", format="mixed", dayfirst=True)
        except TypeError:
            return pd.to_datetime(s, errors="coerce", dayfirst=True, infer_datetime_format=True)

    def _clean_dates_generic(series):
        """Nettoie dates: supprime heures seules, '0000-00-00', vides; gère numéros Excel."""
        s = series.copy()
        s_str = s.astype(str).str.strip().str.lower()

        time_only = s_str.str.fullmatch(r"\d{1,2}:\d{2}(:\d{2})?")
        zero_date = s_str.str.fullmatch(r"(0{1,2}[/\-]0{1,2}[/\-]0{2,4}|0000-00-00)")
        empties   = s_str.isin(["", "nan", "nat", "none", "nul", "null"])

        s = s.mask(time_only | zero_date | empties, np.nan)
        dt = _to_datetime_mixed(s)

        # Numéros Excel éventuels
        serial = pd.to_numeric(s_str, errors="coerce")
        serial_mask = dt.isna() & serial.notna() & serial.between(1, 60000)
        if serial_mask.any():
            dt_serial = pd.to_datetime(serial[serial_mask], unit="D",
                                       origin="1899-12-30", errors="coerce")
            dt.loc[serial_mask] = dt_serial
        return dt

    # ---------- 0) Sécuriser DATE_COLLECTE et dérivées ----------
    df[collect_col] = _to_datetime_mixed(df[collect_col])
    if year_collect_col not in df.columns:
        df[year_collect_col] = df[collect_col].dt.year
    if month_collect_col not in df.columns:
        df[month_collect_col] = df[collect_col].dt.month

    # ---------- 1) Nettoyage PRISE DE SERVICE ----------
    df["PRISE_DE_SERVICE_CLEAN"] = _clean_dates_generic(df[start_col_raw])

    # ---------- 2) Diagnostics ----------
    nb_dates_par_matricule = df.groupby(matricule_col)["PRISE_DE_SERVICE_CLEAN"].nunique(dropna=True)
    matricules_problemes = nb_dates_par_matricule[nb_dates_par_matricule > 1].index.tolist()
    anomalies_dates = (
        df[df[matricule_col].isin(matricules_problemes)]
        [[matricule_col, collect_col, "PRISE_DE_SERVICE_CLEAN"]]
        .sort_values([matricule_col, collect_col])
    )

    # ---------- 3) PRISE_DE_SERVICE_CORRIGEE ----------
    if recompute_corrected or (start_col_corrected not in df.columns):
        panel_sorted = df.sort_values([matricule_col, collect_col], ascending=[True, False])
        ps_corr = (
            panel_sorted
            .drop_duplicates(subset=[matricule_col], keep="first")
            [[matricule_col, "PRISE_DE_SERVICE_CLEAN"]]
            .rename(columns={"PRISE_DE_SERVICE_CLEAN": start_col_corrected})
        )
        if start_col_corrected in df.columns:
            df.drop(columns=[start_col_corrected], inplace=True)
        df = df.merge(ps_corr, on=matricule_col, how="left", validate="many_to_one")

    df[start_col_corrected] = _to_datetime_mixed(df[start_col_corrected])

    # ---------- 4) AGE_AU_SERVICE ----------
    df["AGE_AU_SERVICE"] = pd.NA
    df["AGE_AU_SERVICE_VALIDE"] = pd.NA

    mask = df["AGE_IMPUTE"].notna() & df[start_col_corrected].notna()
    if mask.any():
        # Calcul approximatif en années
        delta_days = (df.loc[mask, collect_col] - df.loc[mask, start_col_corrected]).dt.days
        df.loc[mask, "AGE_AU_SERVICE"] = (df.loc[mask, "AGE_IMPUTE"] - (delta_days // days_per_year)).astype("Float64")
    
    # Bornage
    df["AGE_AU_SERVICE_VALIDE"] = df["AGE_AU_SERVICE"].where(
        (df["AGE_AU_SERVICE"].ge(age_min)) & (df["AGE_AU_SERVICE"].le(age_max))
    )


    # ---------- 5) Imputation par médiane (ANNEE×MOIS×SEXE_IMPUTE) ----------
    group_keys = [year_collect_col, month_collect_col, sex_col]  # toujours utiliser ces trois clés

    # Calcul de la médiane par groupe
    med_group = df.groupby(group_keys)["AGE_AU_SERVICE_VALIDE"].transform("median")
    
    # fallback globale si la médiane de groupe est NaN
    global_med = df["AGE_AU_SERVICE_VALIDE"].median()
    df["AGE_AU_SERVICE_IMPUTE"] = df["AGE_AU_SERVICE_VALIDE"].where(
        df["AGE_AU_SERVICE_VALIDE"].notna(), med_group.fillna(global_med)
    ).round(0)

    # ---------- 6) Report / tableaux ----------
    if return_tables:
        tab_valide = df["AGE_AU_SERVICE_VALIDE"].value_counts(dropna=False).sort_index()
        tab_valide_pct = df["AGE_AU_SERVICE_VALIDE"].value_counts(normalize=True, dropna=False).sort_index()
        tab_impute = df["AGE_AU_SERVICE_IMPUTE"].value_counts(dropna=False).sort_index()
        tab_impute_pct = df["AGE_AU_SERVICE_IMPUTE"].value_counts(normalize=True, dropna=False).sort_index()

        report = {
            "n_matricules_multi_service": len(matricules_problemes),
            "anomalies_dates_head": anomalies_dates.head(sample_anomalies),
            "tab_age_service_valide": pd.DataFrame({
                "Effectif": tab_valide,
                "Pourcentage (%)": (tab_valide_pct * 100).round(2)
            }),
            "tab_age_service_impute": pd.DataFrame({
                "Effectif": tab_impute,
                "Pourcentage (%)": (tab_impute_pct * 100).round(2)
            }),
        }
        return df, report

    return df

### CREATION DE LA VARIBALE ANCIENNTE DANS L'ORGANISME ET DEPUIS LA PRISE DE FONCTION

In [11]:
def build_anciennete(
    df: pd.DataFrame,
    matricule_col="MATRICULE",
    org_col="CODE_ORGANISME",
    collect_col="DATE_COLLECTE",
    age_impute_col="AGE_IMPUTE",
    age_service_impute_col="AGE_AU_SERVICE_IMPUTE",
    days_per_year=365,
    clip_to_function=True,
    broadcast_latest_org=False,  # désactivé pour l'instant
    broadcast_latest_fonction=False,  # désactivé pour l'instant
    do_impute_org=False,
    keep_raw=True,
    return_tables=True
):
    """
    Calcul de l'ancienneté d'un agent dans son organisme et depuis sa fonction.

    Étapes principales :
      1) Ancienneté depuis la fonction
      2) Date d'entrée brute dans l'organisme
      3) Ajustement de la date d'entrée selon AGE_AU_SERVICE_IMPUTE
      4) Calcul ancienneté organisme
      5) Clip à l'ancienneté depuis la fonction si demandé
      6) Diffusion des dernières valeurs par matricule+organisme (désactivée ici)
      7) Imputation optionnelle de l'ancienneté organisme
      8) Génération d'un rapport récapitulatif
    """

    # Travailler sur une copie pour ne pas modifier le DataFrame d'origine
    df = df.copy()

    # ---------- Fonctions utilitaires ----------
    def _to_dt(s):
        """Convertit une série en datetime avec tolérance d'erreurs."""
        try:
            return pd.to_datetime(s, errors="coerce", format="mixed", dayfirst=True)
        except TypeError:
            return pd.to_datetime(s, errors="coerce", dayfirst=True, infer_datetime_format=True)

    def _years_floor(d1, d0):
        """Calcule la différence en années arrondie à l'entier inférieur, sans valeurs négatives."""
        mask = d1.notna() & d0.notna()
        out = pd.Series(pd.NA, index=d1.index, dtype="Float64")
        if mask.any():
            # Calcul de la différence en jours puis conversion en années
            out.loc[mask] = ((d1[mask] - d0[mask]).dt.days // days_per_year).astype("Float64")
        return out.where(out.isna() | (out >= 0))  # On évite les valeurs négatives

    # ---------- Vérification des colonnes identifiants ----------
    # S'assurer qu'il n'y a pas de valeurs manquantes pour le regroupement
    if df[matricule_col].isna().any() or df[org_col].isna().any():
        raise ValueError(
            f"Des valeurs manquantes ont été détectées dans {matricule_col} ou {org_col}. "
            "Corrigez vos données avant de poursuivre."
        )

    # Vérification que la colonne de date de collecte existe
    if collect_col not in df.columns:
        raise KeyError(f"Colonne '{collect_col}' absente.")
    df[collect_col] = _to_dt(df[collect_col])  # Conversion en datetime

    # ---------- 1) Ancienneté depuis la fonction ----------
    # Vérification des colonnes nécessaires pour le calcul
    if age_impute_col not in df.columns or age_service_impute_col not in df.columns:
        raise KeyError(f"Colonnes '{age_impute_col}' et/ou '{age_service_impute_col}' manquantes.")

    # Calcul de l'ancienneté depuis la fonction
    # Différence entre âge actuel imputé et âge au début de la fonction
    df["ANCIENNETE_DEPUIS_FONCTION"] = (
        df[age_impute_col].astype("Float64") - df[age_service_impute_col].astype("Float64")
    ).where(lambda x: x >= 0).round(0)  # Négatifs → NaN

    # ---------- 2) Date d'entrée brute dans l'organisme ----------
    # La plus ancienne date de collecte pour chaque couple (matricule, organisme)
    df["DATE_ENTREE_ORGANISME_RAW"] = (
        df.groupby([matricule_col, org_col])[collect_col].transform("min")
    )

    # ---------- 3) Ajustement de la date d'entrée selon AGE_AU_SERVICE_IMPUTE ----------
    # On ajuste la date brute pour qu'elle ne soit pas antérieure à la date théorique calculée depuis AGE_AU_SERVICE_IMPUTE
    df["DATE_ENTREE_ORGANISME_ADJ"] = df["DATE_ENTREE_ORGANISME_RAW"].where(
        # Condition : si la date brute est >= date théorique → garder la date brute
        df["DATE_ENTREE_ORGANISME_RAW"] >= (df[collect_col] - pd.to_timedelta(df[age_service_impute_col] * 365, unit="D")),
        # Sinon, remplacer par la date calculée depuis AGE_AU_SERVICE_IMPUTE
        df[collect_col] - pd.to_timedelta(df[age_service_impute_col] * 365, unit="D")
    )

    # ---------- 4) Ancienneté organisme ----------
    # Calcul en années à partir de la date ajustée
    df["ANCIENNETE_ORGANISME"] = _years_floor(df[collect_col], df["DATE_ENTREE_ORGANISME_ADJ"])
    if keep_raw:
        # On garde aussi l'ancienneté basée sur la date brute pour contrôle
        df["ANCIENNETE_ORGANISME_RAW"] = _years_floor(df[collect_col], df["DATE_ENTREE_ORGANISME_RAW"])

    # ---------- 5) Clip à l'ancienneté depuis la fonction ----------
    if clip_to_function:
        mask_clip = df["ANCIENNETE_ORGANISME"].notna() & df["ANCIENNETE_DEPUIS_FONCTION"].notna() & \
                    (df["ANCIENNETE_ORGANISME"] > df["ANCIENNETE_DEPUIS_FONCTION"])
        df.loc[mask_clip, "ANCIENNETE_ORGANISME"] = df.loc[mask_clip, "ANCIENNETE_DEPUIS_FONCTION"]

    # ---------- 6) Diffusion des dernières valeurs ----------
    # Pour le moment désactivée
    # broadcast_latest_org et broadcast_latest_fonction = False

    # ---------- 7) Imputation optionnelle de l'ancienneté organisme ----------
    if do_impute_org:
        # Vérification que la colonne sexe imputé existe
        if "SEXE_IMPUTE" not in df.columns:
            raise KeyError("La colonne 'SEXE_IMPUTE' est absente. Imputation par sexe impossible.")
        # Clés pour le calcul de la médiane (année, mois, sexe)
        group_keys = ["ANNEE_COLLECTE", "MOIS_COLLECTE", "SEXE_IMPUTE"]
        # Calcul de la médiane par groupe
        med_org = df.groupby(group_keys)["ANCIENNETE_ORGANISME"].transform("median")
        # Remplacement des valeurs manquantes par la médiane
        df["ANCIENNETE_ORGANISME_IMPUTE"] = df["ANCIENNETE_ORGANISME"].where(
            df["ANCIENNETE_ORGANISME"].notna(), med_org
        ).round(0)

    # ---------- 8) Report optionnel ----------
    if return_tables:
        rep = {
            "nb_lignes_collecte_manquante": int(df[collect_col].isna().sum()),
            "tab_anc_org": df["ANCIENNETE_ORGANISME"].value_counts(dropna=False).sort_index(),
            "tab_anc_fonction": df["ANCIENNETE_DEPUIS_FONCTION"].value_counts(dropna=False).sort_index(),
        }
        if keep_raw:
            rep["tab_anc_org_raw"] = df["ANCIENNETE_ORGANISME_RAW"].value_counts(dropna=False).sort_index()
        if do_impute_org:
            rep["tab_anc_org_impute"] = df["ANCIENNETE_ORGANISME_IMPUTE"].value_counts(dropna=False).sort_index()
        return df, rep

    return df

# CHARGEMENT DE LA BASE DE TRAVAIL

In [12]:
# Connexion S3
s3_client = boto3.client(
 "s3",
 endpoint_url="http://minio.mon-namespace.svc.cluster.local:80", # service interne K8s
 aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
 aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
 verify=False # à désactiver si SSL auto-signé ; sinon mettre True avec certificat valide
)

# Paramètres
bucket_name = "admindataanstat"

# object_key 
object_key = "Solde/panel_solde_df_1_code.parquet"  # Chemin dans le bucket

# Télécharger l'objet depuis S3
obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)

# Charger dans un DataFrame depuis le fichier Parquet
panel_solde_df = pd.read_parquet(io.BytesIO(obj['Body'].read()))

# Afficher un aperçu
panel_solde_df.head()

NameError: name 'boto3' is not defined

In [12]:
import pandas as pd
path = r"C:\Users\a_coulibaly\OneDrive - Metrics & Decisions\INS\CAE\Banque de données travailleurs\Data\Solde\panel_solde_df_1_code.parquet"

panel_solde_df = pd.read_parquet(path, engine="fastparquet")  # <-- alternative à pyarrow

# Afficher un aperçu
panel_solde_df.head()

,MATRICULE||CODE_ORGANISME,MATRICULE,NOM,DATE NAISSANCE,SEXE,SITUATION MATRIMONIALE,NOMBRE ENFANT,LIEU AFFECTATION,SERVICE,ORGANISME,...,SERVICE_NORM,CODE_SERVICE,ORGANISME_NORM,CODE_ORGANISME,CLASSE/ECHELON_NORM,CODE_CLASSE/ECHELON,EMPLOI_NORM,CODE_EMPLOI,SITUATION_NORM,CODE_SITUATION
0,011242X15,011242X,DOSSO MEGBO,1939-01-01,Masculin,Marié,0.0,Seguela,D. G. A. A. T.,Min. d'Etat Administration du Territoire,...,d g a a t,1514,min d etat administration du territoire,15,,None,,None,en activite,10
1,012856Q15,012856Q,KHISSY BEYNIOUAH FULBERT,1939-01-01,Masculin,Célibataire,0.0,Bouaké,D. G. A. A. T.,Min. d'Etat Administration du Territoire,...,d g a a t,1514,min d etat administration du territoire,15,,None,,None,en activite,10
2,013454N15,013454N,VAHA TOMAS GNOMBLEI HENRI,1924-01-01,Masculin,Marié,0.0,Guiglo,D. G. A. A. T.,Min. d'Etat Administration du Territoire,...,d g a a t,1514,min d etat administration du territoire,15,,None,,None,en activite,10
3,027861L25,027861L,CAPET YATO MATHIEU,1943-03-01,Masculin,Marié,0.0,Abidjan,A affecter,"Min. Affaires Etrangères, de l'Intégration Afr...",...,a affecter,1800,"min affaires etrangeres, de l integration afri...",25,,None,,None,en activite,10
4,039659M38,039659M,COULIBALY YOUSSOUF,1945-12-01,Masculin,Marié,2.0,Abidjan,Dir Personnel Police Nationale,Min. de l'Intérieur et de la Sécurité,...,dir personnel police nationale,3813,min de l interieur et de la securite,38,commissaire divis 1er ech,LW,commissaire de police,P1ZC,retraite pour limite age,60


In [13]:
print (panel_solde_df.columns)
print(f"Nous avons {len(panel_solde_df)} observations")

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'FONCTION_NORM',
       'CODE_FONCTION', 'SERVICE_NORM', 'CODE_SERVICE', 'ORGANISME_NORM',
       'CODE_ORGANISME', 'CLASSE/ECHELON_NORM', 'CODE_CLASSE/ECHELON',
       'EMPLOI_NORM', 'CODE_EMPLOI', 'SITUATION_NORM', 'CODE_SITUATION'],
      dtype='object')
Nous avons 15627963 observations


# AFFICHAGE DES DIFFERENTES VARIABLES CREES 

## CATEGORIE 

In [16]:
# Appliquer la fonction de création des catégories 1 et 2
panel_solde_df = build_categories(panel_solde_df, grade_col="GRADE", grade2_col="GRADE 2")

# Aperçu
panel_solde_df[["GRADE","CATEGORIE_1","GRADE 2","CATEGORIE_2"]].head()

,GRADE,CATEGORIE_1,GRADE 2,CATEGORIE_2
0,Non Fonctionnaire Personnel à Décision Provisoire,Non Fonctionnaire,60,Non Fonctionnaire
1,Non Fonctionnaire Personnel à Décision Provisoire,Non Fonctionnaire,60,Non Fonctionnaire
2,Non Fonctionnaire Personnel à Décision Provisoire,Non Fonctionnaire,60,Non Fonctionnaire
3,Non Fonctionnaire Personnel à Décision Provisoire,Non Fonctionnaire,60,Non Fonctionnaire
4,Fonctionnaire Catégorie A7,A,A7,A


### Extraction des modalités des variables GRADE et GRADE 2

In [17]:
# Étape 1 : Nettoyer les données
unique_grades = panel_solde_df[["GRADE", "CATEGORIE_1"]].dropna()  # Supprime les NaN
unique_grades = unique_grades[(unique_grades["GRADE"] != "") & (unique_grades["CATEGORIE_1"] != "")]  # Supprime les vides

# Étape 2 : Extraire les lignes uniques
unique_grades = unique_grades.drop_duplicates().reset_index(drop=True)

# Étape 3 : Sauvegarde dans Excel
unique_grades.to_excel("unique_grades.xlsx", index=False)


In [18]:
# Étape 1 : Nettoyer les données
unique_grades = panel_solde_df[["GRADE 2", "CATEGORIE_2"]].dropna()  # Supprime les NaN
unique_grades = unique_grades[(unique_grades["GRADE 2"] != "") & (unique_grades["CATEGORIE_2"] != "")]  # Supprime les vides

# Étape 2 : Extraire les lignes uniques
unique_grades = unique_grades.drop_duplicates().reset_index(drop=True)

# Étape 3 : Sauvegarde dans Excel
unique_grades.to_excel("unique_grades_2.xlsx", index=False)

In [19]:
# 📊 Créer un tableau croisé des effectifs
tableau_croise_effectifs = pd.crosstab(
    panel_solde_df["CATEGORIE_1"],   # Variable en ligne
    panel_solde_df["CATEGORIE_2"],   # Variable en colonne
    margins=True,        # Ajoute les totaux
    margins_name="Total" # Nom de la ligne/colonne total
)
tableau_croise_effectifs

CATEGORIE_2,Non Fonctionnaire,A,B,C,D,Total
CATEGORIE_1,,,,,,
Non Fonctionnaire,320356,43,0,0,0,320399
A,0,4797789,0,0,0,4797789
B,0,0,6119667,0,0,6119667
C,0,0,0,3961427,0,3961427
D,0,0,0,0,428681,428681
Total,320356,4797832,6119667,3961427,428681,15627963


In [20]:
print(((panel_solde_df["CATEGORIE_1"].isna()) | (panel_solde_df["CATEGORIE_1"].str.strip() == "")).sum())
print(((panel_solde_df["CATEGORIE_2"].isna()) | (panel_solde_df["CATEGORIE_2"].str.strip() == "")).sum())

0
0


In [21]:
filtre = panel_solde_df[
    (panel_solde_df["CATEGORIE_1"] == "Non Fonctionnaire") &
    (panel_solde_df["CATEGORIE_2"] == "A")
]
tab_statut = filtre["EMPLOI_NORM"].value_counts()
print(tab_statut)

EMPLOI_NORM
                         36
medecin generaliste       5
inspecteur des impots     2
Name: count, dtype: int64


## STATUT

In [22]:
# Appliquer la fonction de création de la variable statut
panel_solde_df, rep = build_statut_from_cats(panel_solde_df)
rep

{'repartition_statut': STATUT
 Non Fonctionnaire      315125
 Fonctionnaire        15307347
 Cas particulier          5491
 Name: count, dtype: int64}

In [23]:
filtre = panel_solde_df[
    (panel_solde_df["CATEGORIE_1"] == "Non Fonctionnaire") &
    (panel_solde_df["CATEGORIE_2"] == "A")
]
tab_statut = filtre["STATUT"].value_counts()
print(tab_statut)

STATUT
Cas particulier      43
Non Fonctionnaire     0
Fonctionnaire         0
Name: count, dtype: int64


In [24]:
# Tabulation simple sur STATUT
tab_statut = panel_solde_df["STATUT"].value_counts(dropna=False).sort_index()

# Total lignes
total_lignes = tab_statut.sum()

# Construction d'un DataFrame avec proportions
summary_df = pd.DataFrame({
    "Statut": tab_statut.index,
    "Effectif": tab_statut.values,
    "Proportion (%)": (tab_statut.values / total_lignes * 100).round(2)
})

# Ajouter une ligne Total
summary_df = pd.concat([
    summary_df,
    pd.DataFrame({
        "Statut": ["Total"],
        "Effectif": [total_lignes],
        "Proportion (%)": [summary_df["Proportion (%)"].sum()]
    })
], ignore_index=True)

summary_df

,Statut,Effectif,Proportion (%)
0,Non Fonctionnaire,315125,2.02
1,Fonctionnaire,15307347,97.95
2,Cas particulier,5491,0.04
3,Total,15627963,100.01


## STATUT DEFINITIF

In [25]:
# Appliquer la fonction de création du statut définitif en s'appuyant sur la période MMYYY
panel_solde_df, rep = build_statut_def_periode(panel_solde_df, return_report=True)
rep

{'Statut définifitif': Statut_def_mois
 Non Fonctionnaire      231219
 Cas particulier          4964
 Fonctionnaire        15391780
 Name: count, dtype: int64}

In [26]:
# Récupérer la répartition des lignes par Statut_def
tab_Statut_def = rep["Statut définifitif"]

# Total lignes
total_lignes = tab_Statut_def.sum()

# Construction d'un DataFrame avec proportions
summary_df = pd.DataFrame({
    "Statut Définitif": tab_Statut_def.index,
    "Effectif": tab_Statut_def.values,
    "Proportion (%)": (tab_Statut_def.values / total_lignes * 100).round(2)
})

# Ajouter une ligne Total
summary_df = pd.concat([
    summary_df,
    pd.DataFrame({
        "Statut Définitif": ["Total"],
        "Effectif": [total_lignes],
        "Proportion (%)": [summary_df["Proportion (%)"].sum()]
    })
], ignore_index=True)

summary_df

,Statut Définitif,Effectif,Proportion (%)
0,Non Fonctionnaire,231219,1.48
1,Cas particulier,4964,0.03
2,Fonctionnaire,15391780,98.49
3,Total,15627963,100.00


In [27]:
filtre = panel_solde_df[
    (panel_solde_df["CATEGORIE_1"] == "Non Fonctionnaire") &
    (panel_solde_df["CATEGORIE_2"] == "A")
]
tab_statut = filtre["Statut_def_mois"].value_counts()
print(tab_statut)

Statut_def_mois
Cas particulier      43
Non Fonctionnaire     0
Fonctionnaire         0
Name: count, dtype: int64


In [28]:
# Filtrer les lignes "Cas particulier"
cas_particuliers_df = panel_solde_df[panel_solde_df["Statut_def_mois"] == "Cas particulier"]

# Vérifier si un même matricule apparaît plusieurs fois dans la même période
doublons = cas_particuliers_df[cas_particuliers_df.duplicated(subset=["MATRICULE", "PERIODE"], keep=False)]

if doublons.empty:
    print("✅ Chaque matricule est unique par période.")
else:
    print(f"⚠️ {len(doublons)} doublons trouvés (mêmes matricules dans la même période).")
    print(doublons[["MATRICULE", "PERIODE"]].sort_values(by=["MATRICULE", "PERIODE"]))

⚠️ 2 doublons trouvés (mêmes matricules dans la même période).
        MATRICULE  PERIODE
9228788   089081R   102018
9228789   089081R   102018


In [29]:
# Filtrer les lignes "Cas particulier"
cas_particuliers_df = panel_solde_df[panel_solde_df["Statut_def_mois"] == "Cas particulier"]

# Trier par la colonne "MATRICULE||CODE_ORGANISME"
cas_particuliers_df = cas_particuliers_df.sort_values(by="MATRICULE")

# Chemin et nom du fichier Excel de sortie
output_file = "cas_particuliers_def_periode.xlsx"

# Exporter avec index
cas_particuliers_df[["MATRICULE", "MATRICULE||CODE_ORGANISME", "EMPLOI_NORM", "CATEGORIE_1", "CATEGORIE_2", "PERIODE"]].to_excel(output_file, index=False)

print(f"Fichier exporté : {output_file} avec {len(cas_particuliers_df)} observations")

Fichier exporté : cas_particuliers_def_periode.xlsx avec 4964 observations


In [30]:
panel_solde_df.columns

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'FONCTION_NORM',
       'CODE_FONCTION', 'SERVICE_NORM', 'CODE_SERVICE', 'ORGANISME_NORM',
       'CODE_ORGANISME', 'CLASSE/ECHELON_NORM', 'CODE_CLASSE/ECHELON',
       'EMPLOI_NORM', 'CODE_EMPLOI', 'SITUATION_NORM', 'CODE_SITUATION',
       'CATEGORIE_1', 'CATEGORIE_2', 'STATUT', 'Statut_def_mois'],
      dtype='object')

In [31]:
# Appliquer la fonction de création du statut définitif en s'appuyant sur l'année
panel_solde_df, rep = build_statut_def_annee(panel_solde_df, return_report=True)
rep

{'Statut_def_distribution': Statut_def_annee
 Non Fonctionnaire      230930
 Cas particulier          3469
 Fonctionnaire        15393564
 Name: count, dtype: int64}

In [32]:
# Récupérer la répartition des lignes par Statut_def
tab_Statut_def = rep["Statut_def_distribution"]

# Total lignes
total_lignes = tab_Statut_def.sum()

# Construction d'un DataFrame avec proportions
summary_df = pd.DataFrame({
    "Statut Définitif": tab_Statut_def.index,
    "Effectif": tab_Statut_def.values,
    "Proportion (%)": (tab_Statut_def.values / total_lignes * 100).round(2)
})

# Ajouter une ligne Total
summary_df = pd.concat([
    summary_df,
    pd.DataFrame({
        "Statut Définitif": ["Total"],
        "Effectif": [total_lignes],
        "Proportion (%)": [summary_df["Proportion (%)"].sum()]
    })
], ignore_index=True)

summary_df

,Statut Définitif,Effectif,Proportion (%)
0,Non Fonctionnaire,230930,1.48
1,Cas particulier,3469,0.02
2,Fonctionnaire,15393564,98.50
3,Total,15627963,100.00


In [33]:
filtre = panel_solde_df[
    (panel_solde_df["CATEGORIE_1"] == "Non Fonctionnaire") &
    (panel_solde_df["CATEGORIE_2"] == "A") & 
    (panel_solde_df["EMPLOI_NORM"] != "") 

]
tab_statut = filtre["Statut_def_annee"].value_counts()
print(tab_statut)

Statut_def_annee
Fonctionnaire        7
Non Fonctionnaire    0
Cas particulier      0
Name: count, dtype: int64


In [34]:
# Filtrer les lignes "Cas particulier"
cas_particuliers_df = panel_solde_df[panel_solde_df["Statut_def_annee"] == "Cas particulier"]

# Trier par la colonne "MATRICULE||CODE_ORGANISME"
cas_particuliers_df = cas_particuliers_df.sort_values(by="MATRICULE")

# Chemin et nom du fichier Excel de sortie
output_file = "cas_particuliers_def_annee.xlsx"

# Exporter avec index
cas_particuliers_df[["MATRICULE", "MATRICULE||CODE_ORGANISME", "EMPLOI_NORM", "CATEGORIE_1", "CATEGORIE_2", "ANNEE"]].to_excel(output_file, index=False)

print(f"Fichier exporté : {output_file} avec {len(cas_particuliers_df)} observations")

Fichier exporté : cas_particuliers_def_annee.xlsx avec 3469 observations


## SEXE

In [20]:
panel_solde_df, report = imputer_sexe(
    panel_solde_df,
    sex_col="SEXE",
    collect_col="DATE_COLLECTE",
    sex_valid_col="SEXE_IMPUTE",
    debug=True
)
report

=== Statistiques avant imputation ===


,Effectif
SEXE,
Féminin,4683169
Masculin,10944644
None,150



=== Statistiques après imputation ===


,Effectif
SEXE_IMPUTE,
Féminin,4683169
Masculin,10944794



=== Pourcentages après imputation ===


,Pourcentage (%)
SEXE_IMPUTE,
Féminin,29.966599
Masculin,70.033401



=== Tableau croisé avant/après ===


SEXE_IMPUTE,Féminin,Masculin,Total
SEXE,,,
Féminin,4683169,0,4683169.0
Masculin,0,10944644,10944644.0
NaN,0,150,NaN
Total,4683169,10944794,15627963.0


{'tab_sexe_avant': SEXE
 Féminin      4683169
 Masculin    10944644
 None             150
 Name: count, dtype: int64,
 'tab_sexe_apres': SEXE_IMPUTE
 Féminin      4683169
 Masculin    10944794
 Name: count, dtype: int64,
 'tab_sexe_apres_pct': SEXE_IMPUTE
 Féminin     29.966599
 Masculin    70.033401
 Name: proportion, dtype: float64,
 'crosstab_sexe': SEXE_IMPUTE  Féminin  Masculin       Total
 SEXE                                      
 Féminin      4683169         0   4683169.0
 Masculin           0  10944644  10944644.0
 NaN                0       150         NaN
 Total        4683169  10944794  15627963.0}

In [18]:
variation, details, panel_solde_df, stats = verifier_variation_sexe(
    panel_solde_df, id_col="MATRICULE", sex_col="SEXE_IMPUTE", collect_col="DATE_COLLECTE"
)

# Voir les stats après mise à jour
stats


Nombre d'individus dont le sexe varie dans le temps : 2523


{'tab_avant': SEXE_IMPUTE
 Féminin      4683169
 Masculin    10944794
 Name: count, dtype: int64,
 'tab_apres': SEXE_IMPUTE
 Féminin      4684666
 Masculin    10943297
 Name: count, dtype: int64,
 'tab_apres_pct': SEXE_IMPUTE
 Féminin     29.976178
 Masculin    70.023822
 Name: proportion, dtype: float64}

In [18]:
print(panel_solde_df.columns)

AttributeError: 'tuple' object has no attribute 'columns'

## AGE 

### AGE Valide

In [14]:
panel_solde_df, rep = build_age_valide(
    panel_solde_df,
    strategy="recent",      # ou "mode"
    do_impute_age=True,     # calcule aussi AGE_IMPUTE
    return_tables=True,     # renvoie les tableaux sexe (avant/après)
    debug=False
)

# Afficher les tableaux
print(pd.DataFrame({"Effectif": rep["tab_sexe_avant"], "Pourcentage (%)": rep["tab_sexe_avant_pct"]}))
print(pd.DataFrame({"Effectif": rep["tab_sexe_apres"],  "Pourcentage (%)": rep["tab_sexe_apres_pct"]}))
print(rep["crosstab_sexe"])

# Aperçu des colonnes ajoutées
panel_solde_df[[
    "MATRICULE","DATE_NAISSANCE_CORRIGEE",
    "ANNEE_COLLECTE","MOIS_COLLECTE",
    "SEXE","SEXE_VALIDE",
    "AGE","AGE_VALIDE","AGE_IMPUTE"
]].head()


TypeError: build_age_valide() got an unexpected keyword argument 'strategy'

#### AGE DE PRISE DE SERVICE 

In [21]:
panel_solde_df, rep_srv = build_age_service(
    panel_solde_df,
    matricule_col="MATRICULE",
    start_col_raw="PRISE DE SERVICE",
    start_col_corrected="PRISE_DE_SERVICE_CORRIGEE",
    birth_col="DATE_NAISSANCE_CORRIGEE",
    collect_col="DATE_COLLECTE",
    sex_valid_col="SEXE_VALIDE",
    recompute_corrected=True,   # recalcule la date corrigée depuis la collecte la + récente
    age_min=18, age_max=40,
    days_per_year=365,
    return_tables=True
)

# Aperçu des colonnes ajoutées
panel_solde_df[[
    "MATRICULE","DATE_NAISSANCE_CORRIGEE",
   "AGE_AU_SERVICE_IMPUTE","AGE_IMPUTE"
]].head()


,MATRICULE,DATE_NAISSANCE_CORRIGEE,AGE_AU_SERVICE_IMPUTE,AGE_IMPUTE
0,011242X,1939-01-01,30.0,41.0
1,012856Q,1939-01-01,30.0,41.0
2,013454N,1924-01-01,27.0,41.0
3,027861L,1943-03-01,27.0,41.0
4,039659M,1945-12-01,27.0,69.0


#### ANCIENNETE

In [27]:
# 1) Exécuter la fonction et récupérer le DataFrame enrichi
panel_solde_df, rep_anc = build_anciennete(
    panel_solde_df,
    org_col="ID_ORGANISME",          # ou "CODE_ORGANISME"
    start_service_col="PRISE_DE_SERVICE_CORRIGEE",
    adjust_entry_with_service=True,  # cohérence: entrée org >= prise de service
    clip_to_function=True,           # sécurité
    broadcast_latest_org=True,       # crée ANCIENNETE_ORGANISME_MAJ (dernière valeur du couple)
    keep_raw=True,
    return_tables=True
)

# 2) Aperçu (la colonne ANCIENNETE_ORGANISME_MAJ existe désormais même si certains groupes n'ont pas de collecte)
print("Lignes avec DATE_COLLECTE manquante :", rep_anc["nb_lignes_collecte_manquante"])
print(rep_anc["tab_anc_org"].head())

panel_solde_df[[
    "MATRICULE","ID_ORGANISME","DATE_COLLECTE",
    "DATE_ENTREE_ORGANISME","ANCIENNETE_ORGANISME","ANCIENNETE_ORGANISME_MAJ",
    "AGE_IMPUTE","AGE_AU_SERVICE_IMPUTE","ANCIENNETE_DEPUIS_FONCTION"
]].head()


Lignes avec DATE_COLLECTE manquante : 0
ANCIENNETE_ORGANISME
0.0    3442839
1.0    3046843
2.0    2719925
3.0    2384605
4.0    2145250
Name: count, dtype: Int64


,MATRICULE,ID_ORGANISME,DATE_COLLECTE,DATE_ENTREE_ORGANISME,ANCIENNETE_ORGANISME,ANCIENNETE_ORGANISME_MAJ,AGE_IMPUTE,AGE_AU_SERVICE_IMPUTE,ANCIENNETE_DEPUIS_FONCTION
0,011242X,15,2015-01-01,2015-01-01,0.0,4.0,41.0,30.0,11.0
1,012856Q,15,2015-01-01,2015-01-01,0.0,4.0,41.0,30.0,11.0
2,013454N,15,2015-01-01,2015-01-01,0.0,4.0,41.0,27.0,14.0
3,027861L,25,2015-01-01,2015-01-01,0.0,2.0,41.0,27.0,14.0
4,039659M,38,2015-01-01,2015-01-01,0.0,4.0,69.0,27.0,42.0
